# Expert validation 2: export papers to PDF

For each paper selected by notebook 01, this notebook writes **two PDFs**:

1. **Paper PDF** (`<paper key>.pdf`): the paper content the ranker sees:
   - Abstract
   - Manuscript (full paper text)
   - Dataset Description
   - Code (syntax highlighted)
   - Cited Papers (with their manuscript and code)

2. **Reviews PDF** (`<paper key>_reviews.pdf`): the peer review record:
   - Paper reminder (ID, title, abstract)
   - All peer reviews with scores
   - The meta-review

The paper key, `<seed and timestamp>__<paper id>`, is the `paper_id` raters use in their ratings
files (see notebook 03).

Pipeline: Python builds Markdown, then pandoc and xelatex make the PDF (on Debian or Ubuntu:
`apt-get install pandoc texlive-xetex fonts-dejavu`).

## Setup & Configuration

In [ ]:
import json
import pathlib
import re
import shutil
import subprocess
import sys
import tempfile
from typing import Any

REPO_ROOT = pathlib.Path("../..").resolve()
sys.path.insert(0, str(REPO_ROOT))
OUT_DIR = REPO_ROOT / "outputs/expert_validation"

# Selection list exported by notebook 01. It also names the ranker run's paper_data.json.
SELECTION_LIST_PATH = OUT_DIR / "selection_list.json"
assert SELECTION_LIST_PATH.exists(), f"Missing {SELECTION_LIST_PATH}: run notebook 01 first"
with open(SELECTION_LIST_PATH) as f:
    selection = json.load(f)

PAPER_DATA_PATH = REPO_ROOT / selection["paper_data_path"]
SELECTION_SET_INDEX = int(selection["selection_set_index"])

# Output directory for PDFs: one folder per selected set.
OUTPUT_DIR = OUT_DIR / "paper_pdfs" / f"set_{SELECTION_SET_INDEX:02d}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Paper rendering controls
# Keep manuscript verbatim so the paper PDF contains all authored sections.
INCLUDE_DATASET_SECTION = True
INCLUDE_CODE_SECTION = True
INCLUDE_CITED_PAPERS_SECTION = True
VERIFY_FULL_MANUSCRIPT_INCLUDED = True

assert PAPER_DATA_PATH.exists(), f"Missing paper_data.json at {PAPER_DATA_PATH}"

print(f"Repo root:      {REPO_ROOT}")
print(f"Paper data:     {PAPER_DATA_PATH.relative_to(REPO_ROOT)}")
print(f"Selection list: {SELECTION_LIST_PATH.relative_to(REPO_ROOT)}")
print(f"Output dir:     {OUTPUT_DIR.relative_to(REPO_ROOT)}")
print("Render config:")
print(f"  include_dataset_section={INCLUDE_DATASET_SECTION}")
print(f"  include_code_section={INCLUDE_CODE_SECTION}")
print(f"  include_cited_papers_section={INCLUDE_CITED_PAPERS_SECTION}")
print(f"  verify_full_manuscript_included={VERIFY_FULL_MANUSCRIPT_INCLUDED}")

## Load Data

In [ ]:
with open(PAPER_DATA_PATH) as f:
    all_papers = json.load(f)
print(f"Loaded {len(all_papers)} papers from paper_data.json")

selection_sets_raw = selection["selection_sets"]


def _set_sort_key(set_name: str) -> tuple[int, str]:
    m = re.search(r"(\d+)", str(set_name))
    return (int(m.group(1)) if m else 10**9, str(set_name))


selection_keys_by_set: dict[str, list[str]] = {
    str(set_name): list(keys)
    for set_name, keys in sorted(selection_sets_raw.items(), key=lambda kv: _set_sort_key(kv[0]))
    if keys
}

if not selection_keys_by_set:
    raise ValueError("Selection list contains no non-empty sets")

all_selected_keys = set()
for set_name, keys in selection_keys_by_set.items():
    all_selected_keys.update(keys)
    print(f"{set_name}: {len(keys)} papers")
print(f"Total selected: {len(all_selected_keys)} unique entries")


def _paper_uid_raw(p: dict[str, Any]) -> str:
    return f"{p.get('source_snapshot_dir', '')}::{p['paper_id']}"


def _paper_uid_abs(p: dict[str, Any]) -> str:
    snap_dir = p.get("source_snapshot_dir", "")
    if snap_dir and not pathlib.Path(snap_dir).is_absolute():
        snap_dir = str((REPO_ROOT / snap_dir).resolve())
    return f"{snap_dir}::{p['paper_id']}"


paper_lookup_by_uid: dict[str, dict[str, Any]] = {}
paper_lookup_by_uid_abs: dict[str, dict[str, Any]] = {}
paper_lookup_by_pid: dict[str, list[dict[str, Any]]] = {}

for p in all_papers:
    uid_raw = _paper_uid_raw(p)
    uid_abs = _paper_uid_abs(p)

    snap_dir = p.get("source_snapshot_dir", "")
    if snap_dir and not pathlib.Path(snap_dir).is_absolute():
        p["_source_snapshot_dir_abs"] = str((REPO_ROOT / snap_dir).resolve())
    else:
        p["_source_snapshot_dir_abs"] = snap_dir

    p["_paper_uid"] = uid_raw

    paper_lookup_by_uid[uid_raw] = p
    paper_lookup_by_uid_abs[uid_abs] = p
    paper_lookup_by_pid.setdefault(p["paper_id"], []).append(p)


def _resolve_selection_key(key: str) -> tuple[str, dict[str, Any]]:
    if key in paper_lookup_by_uid:
        p = paper_lookup_by_uid[key]
        return p["_paper_uid"], p

    if key in paper_lookup_by_uid_abs:
        p = paper_lookup_by_uid_abs[key]
        return p["_paper_uid"], p

    candidates = paper_lookup_by_pid.get(key)
    if candidates is None:
        raise KeyError(f"Selected key not found in paper_data.json: {key}")

    if len(candidates) > 1:
        candidate_uids = [c["_paper_uid"] for c in candidates]
        raise ValueError(
            f"Ambiguous legacy paper_id '{key}' maps to multiple papers across snapshots. "
            f"Use paper_uid keys instead. Candidates: {candidate_uids}"
        )

    p = candidates[0]
    return p["_paper_uid"], p


def _resolve_set(keys: list[str], set_name: str) -> list[str]:
    resolved_uids: list[str] = []
    seen: set[str] = set()
    for key in keys:
        uid, _ = _resolve_selection_key(key)
        if uid in seen:
            raise ValueError(f"Duplicate paper in {set_name}: {uid}")
        resolved_uids.append(uid)
        seen.add(uid)
    return resolved_uids


set_ids_by_name: dict[str, list[str]] = {}
for set_name, keys in selection_keys_by_set.items():
    set_ids_by_name[set_name] = _resolve_set(keys, set_name)

all_selected_ids = {uid for ids in set_ids_by_name.values() for uid in ids}
paper_lookup: dict[str, dict[str, Any]] = {
    uid: paper_lookup_by_uid[uid] for uid in all_selected_ids
}

print("Selection key mode: paper_uid-compatible")
for set_name, ids in set_ids_by_name.items():
    print(f"Resolved {set_name}: {len(ids)}")
print(f"Resolved total unique papers: {len(all_selected_ids)}")

## Resolve Citation Chains & Dataset Descriptions

Uses the same functions that the ranker pipeline uses to assemble paper context.

In [ ]:
from ranker.snapshot_import import (
    build_citation_chains,
    collect_citable_papers,
    collect_dataset_descriptions,
    paper_uid,
)

# Build absolute-path copies for context utilities.
# Keep canonical UID based on original source_snapshot_dir text from paper_data.json.
papers_for_context: list[dict[str, Any]] = []
for p in all_papers:
    p_ctx = dict(p)
    p_ctx["paper_uid"] = p.get("_paper_uid", _paper_uid_raw(p))
    p_ctx["source_snapshot_dir"] = p.get(
        "_source_snapshot_dir_abs", p.get("source_snapshot_dir", "")
    )
    papers_for_context.append(p_ctx)

# Citation chains, resolved within each paper's own run. Ancestors come from all accepted papers of
# the runs (the snapshots must be present); without them, only papers in paper_data.json resolve.
snapshot_dirs = sorted(
    {p["source_snapshot_dir"] for p in papers_for_context if p["source_snapshot_dir"]}
)
try:
    citable = collect_citable_papers(snapshot_dirs, "last")
except FileNotFoundError as e:
    print(f"Snapshots not found ({e}); resolving citations within paper_data.json only.")
    citable = None
chains = build_citation_chains(papers_for_context, citable=citable)
citation_chains_by_uid: dict[str, list[dict[str, Any]]] = {
    p["paper_uid"]: chains.get(paper_uid(p), []) for p in papers_for_context
}

print(f"Citation chains built for {len(citation_chains_by_uid)} paper UIDs")

for uid in sorted(all_selected_ids):
    chain = citation_chains_by_uid.get(uid, [])
    chain_ids = [cp["paper_id"] for cp in chain]
    print(f"  {uid}: {len(chain)} ancestor(s) {chain_ids if chain else ''}")

# Dataset descriptions (per absolute snapshot dir)
dataset_descriptions = collect_dataset_descriptions(papers_for_context)
print(f"\nDataset descriptions for {len(dataset_descriptions)} snapshot(s):")
for snap_dir, descs in dataset_descriptions.items():
    print(f"  {pathlib.Path(snap_dir).name}: {list(descs.keys())}")

## Markdown Builder

In [ ]:
CODE_WRAP_WIDTH = 85  # chars; fits ~6.5in text at \footnotesize DejaVu Sans Mono
WIDE_TABLE_THRESHOLD = 8  # columns; tables wider than this get \scriptsize


def wrap_long_lines(text: str, max_width: int = CODE_WRAP_WIDTH) -> str:
    """Hard-wrap lines longer than max_width characters."""
    result = []
    for line in text.split("\n"):
        while len(line) > max_width:
            break_at = line.rfind(" ", 0, max_width)
            if break_at <= 0:
                break_at = max_width
            result.append(line[:break_at])
            line = line[break_at:].lstrip(" ")
        result.append(line)
    return "\n".join(result)


def ensure_blank_before_tables(text: str) -> str:
    """Insert a blank line before pipe tables that follow a non-blank, non-pipe line."""
    lines = text.split("\n")
    out: list[str] = []
    for i, line in enumerate(lines):
        if (
            line.lstrip().startswith("|")
            and i > 0
            and lines[i - 1].strip() != ""
            and not lines[i - 1].lstrip().startswith("|")
        ):
            out.append("")
        out.append(line)
    return "\n".join(out)


def shrink_wide_tables(text: str, threshold: int = WIDE_TABLE_THRESHOLD) -> str:
    r"""Wrap pipe tables that have more than `threshold` columns in \scriptsize.

    Injects raw LaTeX around wide tables so pandoc renders them smaller.
    """
    lines = text.split("\n")
    out: list[str] = []
    in_wide_table = False

    for line in lines:
        stripped = line.lstrip()
        is_pipe = stripped.startswith("|")

        if is_pipe and not in_wide_table:
            # Count columns by counting | separators in this row
            ncols = stripped.count("|") - 1  # subtract outer pipes
            if ncols >= threshold:
                out.append("")
                out.append(r"\begingroup\scriptsize")
                out.append("")
                in_wide_table = True

        if in_wide_table and not is_pipe:
            # Table ended
            out.append("")
            out.append(r"\endgroup")
            out.append("")
            in_wide_table = False

        out.append(line)

    if in_wide_table:
        out.append("")
        out.append(r"\endgroup")
        out.append("")

    return "\n".join(out)


def shift_headings(text: str, levels: int = 2) -> str:
    """Shift markdown heading levels down by `levels` (e.g., # -> ### when levels=2)."""

    def _replace(m: re.Match) -> str:
        new_level = min(len(m.group(1)) + levels, 6)
        return "#" * new_level + m.group(2)

    return re.sub(r"^(#{1,6})([ \t])", _replace, text, flags=re.MULTILINE)


def format_scores_table(scores: dict) -> str:
    """Format review scores as a small markdown table."""
    if not scores:
        return "*No scores available.*"
    header = "| " + " | ".join(scores.keys()) + " |"
    sep = "|" + "|".join(["---"] * len(scores)) + "|"
    vals = "| " + " | ".join(str(v) for v in scores.values()) + " |"
    return f"{header}\n{sep}\n{vals}"


def _safe_title(paper: dict) -> str:
    """Return the paper title, with a fallback for empty/missing titles."""
    title = (paper.get("title") or "").strip()
    return title if title else "[No Title Provided]"


def _paper_uid(paper: dict) -> str:
    """Canonical UID used across multi-snapshot pools."""
    uid = paper.get("_paper_uid")
    if uid:
        return uid
    return f"{paper.get('source_snapshot_dir', '')}::{paper.get('paper_id', 'unknown')}"


def _paper_seed(paper: dict) -> str:
    """Extract seed tag (e.g. seed_1) from the paper UID or snapshot path."""
    uid = _paper_uid(paper)
    m = re.search(r"(seed_\d+)", uid)
    if m:
        return m.group(1)

    snap = paper.get("_source_snapshot_dir_abs") or paper.get("source_snapshot_dir", "")
    m = re.search(r"(seed_\d+)", str(snap))
    return m.group(1) if m else "unknown_seed"


def _paper_snapshot_name(paper: dict) -> str:
    """Readable snapshot tag for filenames."""
    snap = paper.get("_source_snapshot_dir_abs") or paper.get("source_snapshot_dir", "")
    if not snap:
        return "unknown_snapshot"
    return pathlib.Path(snap).name


def _paper_snapshot_tag(paper: dict) -> str:
    """Compact snapshot tag for filenames, preferring seed+timestamp."""
    snap_name = _paper_snapshot_name(paper)

    # Preferred compact form used in this project.
    m = re.search(r"(seed_\d+_\d{6}_\d{4})", snap_name)
    if m:
        return m.group(1)

    # Fallback to seed only if timestamp suffix isn't present.
    m = re.search(r"(seed_\d+)", snap_name)
    if m:
        return m.group(1)

    return snap_name


def _safe_filename_component(text: str) -> str:
    text = re.sub(r"[^A-Za-z0-9._-]+", "_", (text or "").strip())
    text = text.strip("_")
    return text if text else "unknown"


def _paper_pdf_stem(paper: dict) -> str:
    """File name stem `<seed and timestamp>__<paper id>`: the paper key raters use."""
    snap = _safe_filename_component(_paper_snapshot_tag(paper))
    pid = _safe_filename_component(str(paper.get("paper_id", "unknown")))
    return f"{snap}__{pid}"[:180]


def _section_dataset_description(dataset_descs: dict[str, str]) -> str:
    """Build the Dataset Description section."""
    lines = ["## Dataset Description", ""]
    if not dataset_descs:
        lines.append("No dataset description available for this paper.")
    else:
        for idx, (key, desc) in enumerate(dataset_descs.items(), start=1):
            lines.append(f"**Dataset {idx}: `{key}`**")
            lines.append("")
            processed = ensure_blank_before_tables(desc.strip())
            processed = shrink_wide_tables(processed)
            lines.append(processed)
            lines.append("")
    return "\n".join(lines)


def _full_manuscript_text(paper: dict) -> str:
    """Return manuscript exactly as written (fallback: abstract if missing)."""
    manuscript = paper.get("manuscript", "")
    if manuscript and manuscript.strip():
        return manuscript.strip()
    abstract = paper.get("abstract", "N/A")
    return str(abstract).strip()


def _extract_markdown_headings(text: str) -> list[str]:
    return re.findall(r"(?m)^#{1,6}\s+(.+)$", text or "")


def _section_manuscript(paper: dict) -> str:
    manuscript = _full_manuscript_text(paper)
    # Keep verbatim manuscript text so all authored sections remain intact.
    return f"## Full Manuscript\n\n{manuscript}"


def _format_code_block(code: str) -> str:
    """Wrap code in a fenced block with long lines pre-wrapped."""
    code = wrap_long_lines(code)
    return f"```python\n{code}\n```"


def _section_code(paper: dict) -> str:
    code = paper.get("code", "")
    lines = ["## Code", ""]
    if code and code.strip():
        lines.append(_format_code_block(code))
    else:
        lines.append("*No code was provided for this paper.*")
    return "\n".join(lines)


def _section_cited_papers(citation_chain: list[dict]) -> str:
    lines = ["## Cited Papers", ""]
    if not citation_chain:
        lines.append("*This paper does not cite any other papers.*")
        return "\n".join(lines)

    ordered = list(reversed(citation_chain))  # newest first
    n = len(ordered)
    lines.append(f"This paper builds on **{n}** previously published paper(s) (newest first):")

    for idx, cp in enumerate(ordered, start=1):
        cp_id = cp.get("paper_id", "unknown")
        lines.append("")
        lines.append(f"### Cited Paper {idx} of {n} (`{cp_id}`)")
        lines.append("")
        lines.append(f"**Title:** {_safe_title(cp)}")
        lines.append("")
        lines.append(f"**Abstract:** {cp.get('abstract', 'N/A')}")
        lines.append("")

        cp_manuscript = cp.get("manuscript", "")
        if cp_manuscript:
            lines.append("#### Manuscript")
            lines.append("")
            lines.append(shift_headings(cp_manuscript, levels=4))
            lines.append("")

        cp_code = cp.get("code", "")
        if cp_code and cp_code.strip():
            lines.append("#### Code")
            lines.append("")
            lines.append(_format_code_block(cp_code))
            lines.append("")

        lines.append("---")

    return "\n".join(lines)


def _section_reviews(paper: dict) -> str:
    """Build the Reviews section (used in the reviews PDF)."""
    reviews = paper.get("reviews", [])
    lines = ["## Reviews", ""]
    if not reviews:
        lines.append("*No reviews available for this paper.*")
        return "\n".join(lines)

    lines.append(f"{len(reviews)} review(s):\n")

    for idx, review in enumerate(reviews, start=1):
        reviewer = review.get("reviewer_id", "unknown")
        iteration = review.get("iteration", "N/A")
        scores = review.get("scores", {})

        lines.append(f"### Review {idx} — Reviewer: `{reviewer}`")
        lines.append("")
        lines.append(f"**Iteration:** {iteration}")
        lines.append("")
        lines.append(format_scores_table(scores))
        lines.append("")

        review_text = review.get("review_text", "*No review text.*")
        lines.append(shift_headings(review_text, levels=3))
        lines.append("")
        lines.append("---")
        lines.append("")

    return "\n".join(lines)


def _section_metareview(paper: dict) -> str:
    """Build the Metareview section (last section in reviews PDF)."""
    metareview = paper.get("metareview")
    lines = ["## Metareview", ""]
    if not metareview:
        lines.append("*No metareview available for this paper.*")
        return "\n".join(lines)

    decision = (metareview.get("decision") or "N/A").strip()
    overall_score = metareview.get("overall_score")
    rank = metareview.get("rank", "N/A")

    score_str = f"{overall_score:.2f}" if overall_score is not None else "N/A"
    decision_display = decision.capitalize()

    lines.append(
        f"**Decision:** {decision_display} | **Overall Score:** {score_str} | **Rank:** {rank}"
    )
    lines.append("")

    meta_text = (metareview.get("meta_review_text") or "").strip()
    if meta_text:
        lines.append(shift_headings(meta_text, levels=2))
    else:
        lines.append("*No metareview text provided.*")

    return "\n".join(lines)


# ── Paper PDF (no reviews) ──────────────────────────────────────────────


def build_paper_markdown(
    paper: dict,
    paper_set: str,
    citation_chain: list[dict],
    dataset_descs: dict[str, str],
) -> str:
    """Build paper Markdown with full verbatim manuscript + optional context sections."""
    pid = paper["paper_id"]
    seed = _paper_seed(paper)
    title = _safe_title(paper)

    front = [
        "---",
        f'title: "{title}"',
        f'subtitle: "{pid}"',
        "---",
    ]

    header = [
        "",
        f"**Seed:** `{seed}`",
        "",
        f"**Paper ID:** `{pid}` | "
        f"**Agent:** `{paper.get('primary_agent_id', 'N/A')}` | "
        f"**Iteration:** {paper.get('publication_t', 'N/A')}",
        "",
    ]

    sections = [
        "\n".join(front),
        "\n".join(header),
        _section_manuscript(paper),
    ]

    if INCLUDE_DATASET_SECTION:
        sections.extend([r"\newpage", _section_dataset_description(dataset_descs)])
    if INCLUDE_CODE_SECTION:
        sections.extend([r"\newpage", _section_code(paper)])
    if INCLUDE_CITED_PAPERS_SECTION:
        sections.extend([r"\newpage", _section_cited_papers(citation_chain)])

    paper_md = "\n\n".join(sections)

    if VERIFY_FULL_MANUSCRIPT_INCLUDED:
        manuscript_src = _full_manuscript_text(paper)
        if manuscript_src not in paper_md:
            raise ValueError(
                f"Full manuscript text not included for {paper.get('paper_id', 'unknown')}"
            )

        src_heads = _extract_markdown_headings(manuscript_src)
        if src_heads:
            rendered_heads = set(_extract_markdown_headings(paper_md))
            missing = [h for h in src_heads if h not in rendered_heads]
            if missing:
                raise ValueError(
                    f"Missing manuscript headings for {paper.get('paper_id', 'unknown')}: {missing[:5]}"
                )

    return paper_md


# ── Reviews PDF ─────────────────────────────────────────────────────────


def build_reviews_markdown(paper: dict, paper_set: str) -> str:
    """Build the reviews Markdown (reminder header + all reviews + metareview)."""
    pid = paper["paper_id"]
    seed = _paper_seed(paper)
    title = _safe_title(paper)

    front = [
        "---",
        f'title: "Reviews — {title}"',
        f'subtitle: "{pid}"',
        "---",
    ]

    reminder = [
        "## Paper Reminder",
        "",
        f"**Seed:** `{seed}`",
        "",
        f"**Paper ID:** `{pid}` | "
        f"**Agent:** `{paper.get('primary_agent_id', 'N/A')}` | "
        f"**Iteration:** {paper.get('publication_t', 'N/A')}",
        "",
        f"**Title:** {title}",
        "",
        f"**Abstract:** {paper.get('abstract', 'N/A')}",
    ]

    sections = [
        "\n".join(front),
        "\n".join(reminder),
        r"\newpage",
        _section_reviews(paper),
        r"\newpage",
        _section_metareview(paper),
    ]

    return "\n\n".join(sections)


# Quick sanity check
_first_set_name = next(iter(set_ids_by_name))
_test_uid = set_ids_by_name[_first_set_name][0]
_test_paper = paper_lookup[_test_uid]
_test_chain = citation_chains_by_uid.get(_test_uid, [])
_test_descs = dataset_descriptions.get(
    _test_paper.get("_source_snapshot_dir_abs", _test_paper.get("source_snapshot_dir", "")),
    {},
)

_test_md = build_paper_markdown(_test_paper, _first_set_name, _test_chain, _test_descs)
_test_reviews_md = build_reviews_markdown(_test_paper, _first_set_name)
_test_ms = _full_manuscript_text(_test_paper)
print(f"Paper markdown for {_test_uid}: {len(_test_md)} chars")
print(f"Reviews markdown for {_test_uid}: {len(_test_reviews_md)} chars")
print(f"Test manuscript headings preserved: {len(_extract_markdown_headings(_test_ms))}")

# Check empty-title papers
for uid in sorted(all_selected_ids):
    t = (paper_lookup[uid].get("title") or "").strip()
    if not t:
        print(f"  {uid}: will use '[No Title Provided]'")

## PDF Generation

In [ ]:
# LaTeX preamble:
# - Conditionally redefine Shaded as a no-op to avoid "Dimension too large"
#   errors on very long code blocks.
# - Set code font to \footnotesize for readability at narrower widths.
LATEX_PREAMBLE = r"""\makeatletter
\@ifundefined{endShaded}{}{\renewenvironment{Shaded}{}{}}
\@ifundefined{endHighlighting}{}{%
  \RecustomVerbatimEnvironment{Highlighting}{Verbatim}{%
    fontsize=\footnotesize,commandchars=\\\{\}}%
}
\makeatother
"""

_preamble_path = OUTPUT_DIR / "_preamble.tex"
_preamble_path.write_text(LATEX_PREAMBLE)
print(f"Preamble written to {_preamble_path}")


def generate_pdf(
    markdown_content: str,
    output_path: pathlib.Path,
    timeout: int = 300,
) -> bool:
    """Convert markdown to PDF using pandoc + xelatex."""
    md_fd, md_path = tempfile.mkstemp(suffix=".md", dir=str(OUTPUT_DIR))
    try:
        with open(md_fd, "w") as f:
            f.write(markdown_content)

        if shutil.which("xelatex"):
            pdf_engine = "xelatex"
        elif shutil.which("pdflatex"):
            # Some conda TeX installs expose pdflatex but miss format files.
            fmt = subprocess.run(["kpsewhich", "pdflatex.fmt"], capture_output=True, text=True)
            if fmt.returncode != 0 or not fmt.stdout.strip():
                print("  PANDOC ERROR: pdflatex is present but unusable (missing pdflatex.fmt).")
                print(
                    "  Install xelatex (recommended: sudo apt-get install -y texlive-xetex) and rerun."
                )
                return False
            pdf_engine = "pdflatex"
        else:
            print("  PANDOC ERROR: no PDF engine found (need xelatex or pdflatex).")
            return False

        cmd = [
            "pandoc",
            md_path,
            "-o",
            str(output_path),
            "-f",
            "markdown-shortcut_reference_links",
            f"--pdf-engine={pdf_engine}",
            "--toc",
            "--toc-depth=2",
            "-V",
            "geometry:margin=1in",
            "-V",
            "fontsize=11pt",
            "-V",
            "documentclass=article",
            "-V",
            "colorlinks=true",
            "-V",
            "linkcolor=blue",
            "-H",
            str(_preamble_path),
        ]

        # Font variables are only meaningful for xelatex/lualatex.
        if pdf_engine == "xelatex":
            cmd.extend(
                [
                    "-V",
                    "mainfont=DejaVu Serif",
                    "-V",
                    "sansfont=DejaVu Sans",
                    "-V",
                    "monofont=DejaVu Sans Mono",
                ]
            )

        print(f"  Using PDF engine: {pdf_engine}")

        result = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)
        if result.returncode != 0:
            print(f"  PANDOC ERROR (exit {result.returncode}):")
            print(f"  {result.stderr[-800:]}")
            return False
        return True
    finally:
        pathlib.Path(md_path).unlink(missing_ok=True)

In [ ]:
# Test with one paper before running the full batch
_test_stem = "test__" + _paper_pdf_stem(_test_paper)
_test_out = OUTPUT_DIR / f"{_test_stem}.pdf"
_test_rev_out = OUTPUT_DIR / f"{_test_stem}_reviews.pdf"

print(f"Generating test PDFs for {_test_uid} ...")
ok1 = generate_pdf(_test_md, _test_out)
ok2 = generate_pdf(_test_reviews_md, _test_rev_out)

for label, ok, path in [("Paper", ok1, _test_out), ("Reviews", ok2, _test_rev_out)]:
    if ok:
        sz = path.stat().st_size / 1024
        print(f"  {label}: {path.name} ({sz:.0f} KB)")
    else:
        print(f"  {label}: FAILED")

## Generate All PDFs

In [ ]:
results: dict[str, dict[str, bool]] = {}

for set_name, paper_ids in set_ids_by_name.items():
    print(f"\n{'=' * 60}")
    print(f"  {set_name.upper()} — {len(paper_ids)} papers")
    print(f"{'=' * 60}")

    for uid in paper_ids:
        paper = paper_lookup[uid]
        chain = citation_chains_by_uid.get(uid, [])
        snap_dir_abs = paper.get("_source_snapshot_dir_abs", paper.get("source_snapshot_dir", ""))
        descs = dataset_descriptions.get(snap_dir_abs, {})

        stem = _paper_pdf_stem(paper)

        # Paper PDF
        paper_md = build_paper_markdown(paper, set_name, chain, descs)
        paper_path = OUTPUT_DIR / f"{stem}.pdf"

        # Reviews PDF
        reviews_md = build_reviews_markdown(paper, set_name)
        reviews_path = OUTPUT_DIR / f"{stem}_reviews.pdf"

        print(f"  {uid} ... ", end="", flush=True)
        ok_paper = generate_pdf(paper_md, paper_path)
        ok_reviews = generate_pdf(reviews_md, reviews_path)
        results[uid] = {"paper": ok_paper, "reviews": ok_reviews}

        parts = []
        if ok_paper:
            parts.append(f"paper {paper_path.stat().st_size // 1024}KB")
        else:
            parts.append("paper FAILED")
        if ok_reviews:
            parts.append(f"reviews {reviews_path.stat().st_size // 1024}KB")
        else:
            parts.append("reviews FAILED")
        print(", ".join(parts))

# Summary
n_paper_ok = sum(r["paper"] for r in results.values())
n_reviews_ok = sum(r["reviews"] for r in results.values())
total = len(results)
print(f"\n{'=' * 60}")
print(f"  DONE: papers {n_paper_ok}/{total}, reviews {n_reviews_ok}/{total}")
print(f"{'=' * 60}")

failures = [uid for uid, r in results.items() if not r["paper"] or not r["reviews"]]
if failures:
    print(f"\nFailed: {failures}")

## Cleanup & Summary

In [ ]:
# Remove test PDFs and preamble file
_test_out.unlink(missing_ok=True)
_test_rev_out.unlink(missing_ok=True)
_preamble_path.unlink(missing_ok=True)

# List all generated PDFs
pdfs = sorted(OUTPUT_DIR.glob("*.pdf"))
print(f"Generated {len(pdfs)} PDFs in {OUTPUT_DIR.relative_to(REPO_ROOT)}/")
print()
total_size = 0
for pdf in pdfs:
    sz = pdf.stat().st_size / 1024
    total_size += sz
    print(f"  {pdf.name:55s} {sz:8.0f} KB")
print(f"{'':55s} {'─' * 8}")
print(f"  {'Total':55s} {total_size:8.0f} KB")